In [1]:
import pandas as pd
import numpy as np
import yfinance as yf


In [2]:
tickers = [
    "^NDX",
    "^GSPC",
    "^DJI",
    "^RUT",
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "GOOGL",
    "META",
    "TSLA",
]

prices = yf.download(tickers=tickers, start ="2010-01-01",multi_level_index=False, auto_adjust=True)
prices = prices["Close"]

prices.isnull().sum()
prices.info()


[*********************100%***********************]  11 of 11 completed

<class 'pandas.DataFrame'>
DatetimeIndex: 4227 entries, 2010-01-04 to 2026-10-22
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    4227 non-null   float64
 1   AMZN    4227 non-null   float64
 2   GOOGL   4227 non-null   float64
 3   META    3628 non-null   float64
 4   MSFT    4227 non-null   float64
 5   NVDA    4227 non-null   float64
 6   TSLA    4105 non-null   float64
 7   ^DJI    4227 non-null   float64
 8   ^GSPC   4227 non-null   float64
 9   ^NDX    4227 non-null   float64
 10  ^RUT    4226 non-null   float64
dtypes: float64(11)
memory usage: 396.3 KB


In [3]:
#R dziennie stope zwrotu
daily_return = prices.pct_change(1).dropna()
daily_return

#ogolny zwrot
total_return = (1+daily_return).prod() - 1


#Zannualizowana zmiennosc / wahania
annualized_volatility= daily_return.std() * (252**0.5)
annualized_volatility


#zannualizowany zwrot
years = len(daily_return) /252
annualized_return = (1 + total_return) ** (1/years) -1
annualized_return

#Wskaznik sharpa
risk_free_rate = 0
sharpe_ratio = ((annualized_return/years) - risk_free_rate)/annualized_volatility
sharpe_ratio

#najwiekszy dołek
rolling_max = prices.cummax()
drawdown = (prices - rolling_max) / rolling_max
drawdown = drawdown.min()

r2 = daily_return.corr()**2

market = daily_return["^NDX"]

def SCL(table):
    company = table
    b , a = np.polyfit(market,company,1)
    tabelka = pd.Series({"alfa": a,"beta" : b})
    return tabelka

scl = daily_return.apply(SCL).T


In [9]:
summary_table = pd.DataFrame({
    "total_return" : total_return,
    "annualized_volatility" : annualized_volatility,
    "sharpe_ratio" : sharpe_ratio,
    "max_drawdown" : drawdown,
    "scl_alfa": scl["alfa"],
    "scl_beta": scl["beta"]
    })

summary_table.to_csv("summary_table.csv")

TypeError: write() argument must be str, not DataFrame